In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.7 MoE introduction

A Mixture of Experts replaces one MLP with many, plus a **router** that sends each
token to just a few. The model gains parameters (capacity) without proportional
compute, because only the chosen experts run per token.

In [ ]:
from torch import nn

torch.manual_seed(0)
n_experts, dim = 4, 8
experts = nn.ModuleList([nn.Linear(dim, dim) for _ in range(n_experts)])
router = nn.Linear(dim, n_experts)
x = torch.randn(5, dim)
choice = router(x).argmax(-1)   # top-1 routing
out = torch.stack([experts[choice[i]](x[i]) for i in range(len(x))])
print("each token routed to one expert:", choice.tolist())
print("output shape:", tuple(out.shape))

Sparsity is the whole point: 4 experts of capacity, but each token pays for one.
Real MoEs route to the top-k and add load-balancing so experts stay busy.

In [ ]:
assert tuple(out.shape) == (5, dim)
assert int(choice.max()) < n_experts